# EEG Classification: FTD vs CN — XGBoost with Enhanced Features + LOO-CV

**Pipeline:**
1. Subject-level stratified 3-way split: Train (70%) / Val (15%) / Test (15%)
2. Preprocessing: downsample 500→128 Hz, bandpass filter 0.5–45 Hz
3. Sliding window: 30s window, 15s step (50% overlap), artifact rejection
4. **Enhanced features (306 total):** RBP + SCC + Hjorth + Permutation Entropy + Hemispheric Asymmetry
5. **LOO-CV on training set** with fixed val set for early stopping
6. Train final model on full training set
7. Evaluate on held-out test set
8. Summary: F1, Accuracy, Precision, Recall for LOO-CV and Test

## 1. Imports

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch, resample, butter, filtfilt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

np.random.seed(42)

## 2. Configuration

In [ ]:
# Paths
DATA_DIR  = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'training')
LABEL_CSV = os.path.join(DATA_DIR, 'train_label_mapping.csv')

# Sampling
ORIG_SFREQ = 500
SFREQ      = 128

# Windowing
WIN_SEC   = 30
STEP_SEC  = 15
WIN_SAMP  = WIN_SEC  * SFREQ   # 3840
STEP_SAMP = STEP_SEC * SFREQ   # 1920

# Artifact rejection threshold (µV — common for scalp EEG)
ARTIFACT_THRESH = 150e-6

# Split ratios
TEST_SIZE = 0.15
VAL_SIZE  = 0.15

# Labels — FTD=1, CN=0
LABEL_MAP = {'F': 1, 'C': 0}

# Channel layout (standard 10-20, 19 channels)
CHANNELS = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz',
            'C4','T4','T5','P3','Pz','P4','T6','O1','O2']
# Left-right homologous pairs (indices into CHANNELS list)
LR_PAIRS = [
    (0, 1),   # Fp1 / Fp2
    (2, 6),   # F7  / F8
    (3, 5),   # F3  / F4
    (7, 11),  # T3  / T4
    (8, 10),  # C3  / C4
    (12, 16), # T5  / T6
    (13, 15), # P3  / P4
    (17, 18), # O1  / O2
]

# XGBoost base params (scale_pos_weight added dynamically after split)
XGB_BASE_PARAMS = dict(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

print(f'Window: {WIN_SEC}s | Step: {STEP_SEC}s | Samples/window: {WIN_SAMP}')
print(f'Asymmetry pairs: {len(LR_PAIRS)} L/R pairs × 5 bands = {len(LR_PAIRS)*5} features')

## 3. Helper Functions

### 3a. Preprocessing

In [ ]:
def load_npy(path):
    return np.load(path, allow_pickle=True)


def downsample(eeg, orig_fs, target_fs):
    """Resample EEG (channels, time) from orig_fs to target_fs."""
    target_points = int(eeg.shape[1] * target_fs / orig_fs)
    return resample(eeg, target_points, axis=1)


def bandpass_filter(eeg, sfreq, lo=0.5, hi=45.0):
    """Apply zero-phase 4th-order Butterworth bandpass filter to EEG (channels, time)."""
    b, a = butter(4, [lo / (sfreq / 2), hi / (sfreq / 2)], btype='bandpass')
    return filtfilt(b, a, eeg, axis=1)

### 3b. Feature Extraction

In [ ]:
def _extract_rbp(epoch, sfreq):
    """Relative Band Power. Returns rbp array (5 bands, 19 ch) and flat (95,)."""
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 25), (25, 45)]
    n_channels = epoch.shape[0]
    freqs, psd = welch(epoch, fs=sfreq, nperseg=sfreq * 2, axis=1)
    total_power = psd.sum(axis=1, keepdims=True)
    total_power[total_power == 0] = 1e-10
    rbp = np.zeros((len(bands), n_channels))
    for b, (fmin, fmax) in enumerate(bands):
        idx = (freqs >= fmin) & (freqs <= fmax)
        rbp[b] = psd[:, idx].sum(axis=1) / total_power.flatten()
    return rbp, rbp.flatten()


def _extract_scc(epoch, sfreq):
    """Spectral Coherence Connectivity via CWT. Returns flat (95,)."""
    morlet_freqs = np.array([2, 6, 10, 18, 35])
    wavelet      = 'cmor1.5-1.0'
    scales       = (pywt.central_frequency(wavelet) * sfreq) / morlet_freqs
    n_channels   = epoch.shape[0]

    coeffs = np.zeros((n_channels, len(morlet_freqs), epoch.shape[1]), dtype=np.complex128)
    for ch in range(n_channels):
        cwt_out, _ = pywt.cwt(epoch[ch], scales, wavelet, sampling_period=1 / sfreq)
        coeffs[ch] = cwt_out

    scc = np.zeros((len(morlet_freqs), n_channels))
    for b in range(len(morlet_freqs)):
        seg   = coeffs[:, b, :]
        csd   = seg @ seg.conj().T
        pv    = np.diag(csd).real
        denom = np.sqrt(np.outer(pv, pv))
        denom[denom == 0] = 1e-10
        scc[b] = np.abs(csd / denom).mean(axis=1)
    return scc.flatten()


def _extract_hjorth(epoch):
    """Hjorth parameters (activity, mobility, complexity) per channel. Returns (57,)."""
    activity   = epoch.var(axis=1)                         # (ch,)
    d1         = np.diff(epoch, axis=1)
    activity_d = d1.var(axis=1)
    activity[activity == 0] = 1e-10
    activity_d[activity_d == 0] = 1e-10
    mobility   = np.sqrt(activity_d / activity)            # (ch,)

    d2          = np.diff(d1, axis=1)
    activity_d2 = d2.var(axis=1)
    activity_d2[activity_d2 == 0] = 1e-10
    mobility_d  = np.sqrt(activity_d2 / activity_d)
    complexity  = mobility_d / mobility                    # (ch,)

    return np.concatenate([activity, mobility, complexity])  # (57,)


def _permutation_entropy(x, order=3, delay=1):
    """Permutation entropy for a 1D signal."""
    n = len(x)
    patterns = {}
    count = 0
    for i in range(n - (order - 1) * delay):
        window = x[i : i + order * delay : delay]
        key = tuple(np.argsort(window))
        patterns[key] = patterns.get(key, 0) + 1
        count += 1
    probs = np.array(list(patterns.values()), dtype=float) / count
    return -np.sum(probs * np.log2(probs + 1e-10))


def _extract_permutation_entropy(epoch, order=3, delay=1):
    """Permutation entropy per channel. Returns (19,)."""
    return np.array([_permutation_entropy(epoch[ch], order, delay)
                     for ch in range(epoch.shape[0])])


def _extract_asymmetry(rbp):
    """
    Hemispheric asymmetry index: (L - R) / (L + R) per band per pair.
    rbp: shape (5 bands, 19 channels)
    Returns (40,) — 5 bands × 8 L/R pairs.
    """
    asym = []
    for li, ri in LR_PAIRS:
        l_power = rbp[:, li]   # (5,)
        r_power = rbp[:, ri]   # (5,)
        denom   = l_power + r_power
        denom[denom == 0] = 1e-10
        asym.append((l_power - r_power) / denom)
    return np.array(asym).flatten()  # (8*5,) = (40,)


def extract_features(epoch, sfreq):
    """
    Extract all features from a single 30-second epoch.

    Feature breakdown:
      RBP           :  95  (5 bands × 19 ch)
      SCC           :  95  (5 bands × 19 ch)
      Hjorth        :  57  (activity + mobility + complexity × 19 ch)
      Perm. Entropy :  19  (one per channel)
      Asymmetry     :  40  (5 bands × 8 L/R pairs)
      ─────────────────────
      Total         : 306
    """
    rbp_2d, rbp_flat = _extract_rbp(epoch, sfreq)
    scc_flat         = _extract_scc(epoch, sfreq)
    hjorth_flat      = _extract_hjorth(epoch)
    pen_flat         = _extract_permutation_entropy(epoch)
    asym_flat        = _extract_asymmetry(rbp_2d)
    return np.concatenate([rbp_flat, scc_flat, hjorth_flat, pen_flat, asym_flat])  # (306,)


def build_feature_names():
    band_names = ['Delta','Theta','Alpha','Beta','Gamma']
    names  = [f'RBP_{b}_{ch}' for b in band_names for ch in CHANNELS]
    names += [f'SCC_{b}_{ch}' for b in band_names for ch in CHANNELS]
    names += [f'Hjorth_Act_{ch}' for ch in CHANNELS]
    names += [f'Hjorth_Mob_{ch}' for ch in CHANNELS]
    names += [f'Hjorth_Cmp_{ch}' for ch in CHANNELS]
    names += [f'PEn_{ch}'   for ch in CHANNELS]
    names += [f'Asym_{b}_{CHANNELS[li]}_{CHANNELS[ri]}'
              for li, ri in LR_PAIRS for b in band_names]
    return names

FEATURE_NAMES = build_feature_names()
print(f'Features per epoch: {len(FEATURE_NAMES)}')

### 3c. Pipeline Utilities

In [ ]:
def extract_all_epochs(data_list, sids, sid_labels, split_name=''):
    """Slide windows, reject artifacts, extract features.

    Returns X (n_epochs, 306), y (n_epochs,), groups (n_epochs, — subject id).
    """
    X_list, y_list, g_list = [], [], []
    total_rejected = 0
    if split_name:
        print(f'Extracting {split_name} features...')

    for eeg, sid, label in zip(data_list, sids, sid_labels):
        n_pts    = eeg.shape[1]
        accepted = 0
        rejected = 0
        for start in range(0, n_pts - WIN_SAMP + 1, STEP_SAMP):
            epoch = eeg[:, start : start + WIN_SAMP]
            if np.abs(epoch).max() > ARTIFACT_THRESH:
                rejected += 1
                continue
            X_list.append(extract_features(epoch, SFREQ))
            y_list.append(label)
            g_list.append(sid)
            accepted += 1
        total_rejected += rejected
        if split_name:
            print(f'  {sid}: {accepted} epochs kept, {rejected} rejected')

    X = np.array(X_list)
    y = np.array(y_list)
    g = np.array(g_list)
    if split_name:
        print(f'  → {X.shape[0]} total epochs, {total_rejected} artifact epochs discarded')
    return X, y, g


def aggregate_predictions(model, X, groups, epoch_labels):
    """Average epoch-level P(FTD) per subject → threshold at 0.5."""
    probs = model.predict_proba(X)[:, 1]
    sids, y_true, y_pred, y_prob = [], [], [], []
    for sid in np.unique(groups):
        mask     = groups == sid
        avg_prob = probs[mask].mean()
        sids.append(sid)
        y_true.append(epoch_labels[mask][0])
        y_pred.append(int(avg_prob >= 0.5))
        y_prob.append(avg_prob)
    return np.array(sids), np.array(y_true), np.array(y_pred), np.array(y_prob)


def compute_metrics(y_true, y_pred):
    return {
        'Accuracy' : accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall'   : recall_score(y_true, y_pred, zero_division=0),
        'F1'       : f1_score(y_true, y_pred, zero_division=0),
    }

## 4. Load Data (FTD + CN only)

In [ ]:
df_labels = pd.read_csv(LABEL_CSV)
df = df_labels[df_labels['label'].isin(['F', 'C'])].reset_index(drop=True)
print(f"Subjects — FTD: {(df['label']=='F').sum()}, CN: {(df['label']=='C').sum()}, Total: {len(df)}")

subject_ids, raw_data, labels = [], [], []

for _, row in df.iterrows():
    sid    = row['anonymized_id']
    label  = row['label']
    folder = 'FTD' if label == 'F' else 'CN'
    path   = os.path.join(DATA_DIR, folder, f'{sid}.npy')

    if not os.path.exists(path):
        print(f'  WARNING: missing {path}')
        continue

    eeg = load_npy(path)                         # (19, time @ 500 Hz)
    eeg = downsample(eeg, ORIG_SFREQ, SFREQ)    # (19, time @ 128 Hz)
    eeg = bandpass_filter(eeg, SFREQ)            # 0.5–45 Hz bandpass

    subject_ids.append(sid)
    raw_data.append(eeg)
    labels.append(LABEL_MAP[label])

subject_ids = np.array(subject_ids)
labels      = np.array(labels)
print(f'Loaded {len(subject_ids)} subjects  (FTD={labels.sum()}, CN={(labels==0).sum()})')

## 5. Subject-Level Stratified Split (70 / 15 / 15)

All epochs from one subject remain in the same split — no data leakage.

In [ ]:
# Step 1: hold out 15% as test
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
trainval_idx, test_idx = next(gss1.split(subject_ids, labels, groups=subject_ids))

# Step 2: from 85% remainder, hold out ~17.6% as val (= 15% of total)
val_size_relative = VAL_SIZE / (1 - TEST_SIZE)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size_relative, random_state=42)
sub_tv = subject_ids[trainval_idx]
lab_tv = labels[trainval_idx]
train_sub_idx, val_sub_idx = next(gss2.split(sub_tv, lab_tv, groups=sub_tv))

train_idx = trainval_idx[train_sub_idx]
val_idx   = trainval_idx[val_sub_idx]

def subset(idx):
    return subject_ids[idx], [raw_data[i] for i in idx], labels[idx]

train_sids, train_data, train_labels = subset(train_idx)
val_sids,   val_data,   val_labels   = subset(val_idx)
test_sids,  test_data,  test_labels  = subset(test_idx)

print(f'Train : {len(train_sids)} subjects  (FTD={train_labels.sum()}, CN={(train_labels==0).sum()})')
print(f'Val   : {len(val_sids)} subjects  (FTD={val_labels.sum()}, CN={(val_labels==0).sum()})')
print(f'Test  : {len(test_sids)} subjects  (FTD={test_labels.sum()}, CN={(test_labels==0).sum()})')

assert len(set(train_sids) & set(val_sids)) == 0
assert len(set(train_sids) & set(test_sids)) == 0
assert len(set(val_sids)   & set(test_sids)) == 0
print('Split sanity check passed: all sets are disjoint.')

## 6. Feature Extraction — Sliding Window (30s / 15s step)

Epochs exceeding the artifact threshold (±150 µV on any channel) are discarded.

In [ ]:
X_train, y_train, g_train = extract_all_epochs(train_data, train_sids, train_labels, 'Train')
X_val,   y_val,   g_val   = extract_all_epochs(val_data,   val_sids,   val_labels,   'Validation')
X_test,  y_test,  g_test  = extract_all_epochs(test_data,  test_sids,  test_labels,  'Test')

## 7. Feature Scaling

In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
print(f'Scaling done. Feature matrix shape: {X_train.shape}')

## 8. Class Imbalance Weight

FTD subjects are fewer than CN. `scale_pos_weight` compensates at the epoch level.

In [ ]:
# Compute from actual training epoch counts (more accurate than raw subject counts)
spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'scale_pos_weight = {spw:.3f}  '
      f'(CN epochs: {(y_train==0).sum()}, FTD epochs: {(y_train==1).sum()})')

XGB_PARAMS = {**XGB_BASE_PARAMS, 'scale_pos_weight': spw}

## 9. LOO-CV on Training Set

For each training subject `s_i`:
- Leave `s_i` out as the CV test subject
- Train XGBoost on remaining training subjects, using the **fixed validation set** for early stopping
- Aggregate epoch predictions to a subject-level decision for `s_i`

In [ ]:
loo_true, loo_pred = [], []
n_folds = len(train_sids)

print(f'Running LOO-CV: {n_folds} folds...\n')

for fold_i, left_out_sid in enumerate(train_sids):
    lo_mask = g_train == left_out_sid
    cv_mask = ~lo_mask

    X_cv_tr = X_train[cv_mask]
    y_cv_tr = y_train[cv_mask]
    X_lo    = X_train[lo_mask]
    y_lo    = y_train[lo_mask]

    # Recompute weight for this fold's training subset
    fold_spw = (y_cv_tr == 0).sum() / max((y_cv_tr == 1).sum(), 1)
    fold_params = {**XGB_BASE_PARAMS, 'scale_pos_weight': fold_spw}

    clf = XGBClassifier(**fold_params)
    clf.fit(
        X_cv_tr, y_cv_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    avg_prob = clf.predict_proba(X_lo)[:, 1].mean()
    y_true_s = y_lo[0]
    y_pred_s = int(avg_prob >= 0.5)

    loo_true.append(y_true_s)
    loo_pred.append(y_pred_s)

    label_str = 'FTD' if y_true_s == 1 else 'CN'
    pred_str  = 'FTD' if y_pred_s == 1 else 'CN'
    correct   = '✓' if y_true_s == y_pred_s else '✗'
    print(f'  Fold {fold_i+1:2d}/{n_folds} | Subject: {left_out_sid} | True: {label_str} | Pred: {pred_str} {correct}')

loo_true = np.array(loo_true)
loo_pred = np.array(loo_pred)
print('\nLOO-CV complete.')

### LOO-CV Metrics & Confusion Matrix

In [ ]:
loo_metrics = compute_metrics(loo_true, loo_pred)

print('LOO-CV Results (subject-level, training set only)')
print('=' * 44)
for k, v in loo_metrics.items():
    print(f'  {k:<12}: {v:.3f}')
print('=' * 44)

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(confusion_matrix(loo_true, loo_pred),
                       display_labels=['CN', 'FTD']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('LOO-CV — Subject-Level')
plt.tight_layout()
plt.show()

## 10. Train Final Model on Full Training Set

In [ ]:
final_model = XGBClassifier(**XGB_PARAMS)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)
print(f'\nBest iteration: {final_model.best_iteration}')

## 11. Evaluate on Held-Out Test Set

In [ ]:
_, test_true, test_pred, _ = aggregate_predictions(final_model, X_test, g_test, y_test)
test_metrics = compute_metrics(test_true, test_pred)

print('Test Results (subject-level)')
print('=' * 44)
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v:.3f}')
print('=' * 44)

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(confusion_matrix(test_true, test_pred),
                       display_labels=['CN', 'FTD']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Test Set — Subject-Level')
plt.tight_layout()
plt.show()

## 12. Summary — LOO-CV vs Test Metrics

In [ ]:
summary = pd.DataFrame({
    'Metric'  : list(loo_metrics.keys()),
    'LOO-CV'  : [f'{v:.3f}' for v in loo_metrics.values()],
    'Test Set': [f'{v:.3f}' for v in test_metrics.values()],
})

print('\n' + '=' * 40)
print(' Final Metrics Summary (Subject-Level)')
print(' Task: FTD vs CN')
print('=' * 40)
print(summary.to_string(index=False))
print('=' * 40)

# Bar chart
x = np.arange(len(loo_metrics))
w = 0.35
loo_vals  = list(loo_metrics.values())
test_vals = list(test_metrics.values())

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - w/2, loo_vals,  w, label='LOO-CV',   color='steelblue')
bars2 = ax.bar(x + w/2, test_vals, w, label='Test Set', color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(list(loo_metrics.keys()))
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('XGBoost FTD vs CN — LOO-CV vs Test Metrics (Subject-Level)')
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()